# Phase 3: Preprocessing

**BOL-LPP Reproduction Project**

Transforms the raw datasets from Phase 1 into model-ready sequences.

**Steps:**
1. Merge price, load, and weather on a common hourly index (North zone)
2. Resolve DST duplicate timestamps and interpolate missing values
3. Engineer calendar features and previous load price
4. Split chronologically into 70% train / 15% validation / 15% test
5. Fit scaler on training data only, apply to all splits
6. Build sliding-window sequences

**Outputs:** processed arrays saved to `data/processed/`

See `docs/deviations.md` for the reasoning behind the split strategy and feature timing choices.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

ZONE = "North"

prices = pd.read_csv(RAW_DIR / "ercot_dam_spp_zones_2013_2018.csv")
load = pd.read_csv(RAW_DIR / "ercot_hourly_load_zones_2013_2018.csv")
weather = pd.read_csv(RAW_DIR / "weather_north_2013_2018.csv")

def to_naive_central(series):
    return pd.to_datetime(series, utc=True).dt.tz_convert("America/Chicago").dt.tz_localize(None)

pr = prices[prices["Zone"] == ZONE][["Interval Start", "SPP"]].copy()
pr["ts"] = to_naive_central(pr["Interval Start"])
pr = pr[["ts", "SPP"]].drop_duplicates("ts")

ld = load[load["Zone"] == ZONE][["Interval Start", "Load_MW"]].copy()
ld["ts"] = to_naive_central(ld["Interval Start"])
ld = ld[["ts", "Load_MW"]].drop_duplicates("ts")

wx = weather.copy()
wx["ts"] = pd.to_datetime(wx["time"])
wx = wx.drop(columns=["time"])

df = pr.merge(ld, on="ts", how="inner").merge(wx, on="ts", how="inner")
df = df.sort_values("ts").reset_index(drop=True)

print("Shape:", df.shape)
print("Missing before interpolation:\n", df.isna().sum())

df = df.interpolate(method="linear")

print("\nMissing after interpolation:", df.isna().sum().sum())
print("Range:", df["ts"].min(), "to", df["ts"].max())

Shape: (52578, 7)
Missing before interpolation:
 ts                   0
SPP                  0
Load_MW              1
temperature_2m       0
dewpoint_2m          0
windspeed_10m        0
winddirection_10m    0
dtype: int64

Missing after interpolation: 0
Range: 2013-01-01 00:00:00 to 2018-12-31 23:00:00


### Merge result

52,578 rows spanning 2013-01-01 to 2018-12-31, with no missing values after interpolation.

This is six rows fewer than the 52,584 hourly observations reported in the paper (Section III).
The difference is the six daylight-saving fall-back hours (one per year, 2013-2018), where the
1:00-2:00 am hour occurs twice. Deduplicating by naive timestamp retains the first occurrence
and discards the second. The paper does not state how it handled these hours; retaining both
would require timezone-aware timestamps throughout, which is incompatible with merging against
the timezone-naive weather series.

## 3.2 Feature engineering

The paper's Figure 4 lists ten input features: Load, Previous Load prices, Temperature, Dew
temperature, Wind direction, Wind speed, Hour, Day, Month, Year.

Four of these are calendar components extracted from the timestamp. "Previous Load prices" is
the price series itself, which the sliding-window construction supplies as history. The features
are used in their raw form rather than cyclically encoded, following the paper.

In [2]:
df["hour"] = df["ts"].dt.hour
df["day"] = df["ts"].dt.day
df["month"] = df["ts"].dt.month
df["year"] = df["ts"].dt.year

FEATURES = [
    "Load_MW",
    "SPP",
    "temperature_2m",
    "dewpoint_2m",
    "winddirection_10m",
    "windspeed_10m",
    "hour",
    "day",
    "month",
    "year",
]
TARGET = "SPP"

df = df[["ts"] + FEATURES]

print("Shape:", df.shape)
print("Features:", FEATURES)
print(df.head(3))

Shape: (52578, 11)
Features: ['Load_MW', 'SPP', 'temperature_2m', 'dewpoint_2m', 'winddirection_10m', 'windspeed_10m', 'hour', 'day', 'month', 'year']
                   ts       Load_MW    SPP  temperature_2m  dewpoint_2m  \
0 2013-01-01 00:00:00  10370.322921  23.12             7.7          7.4   
1 2013-01-01 01:00:00  10153.942806  21.16             7.4          7.0   
2 2013-01-01 02:00:00   9989.243532  19.84             5.9          5.1   

   winddirection_10m  windspeed_10m  hour  day  month  year  
0                  3           13.7     0    1      1  2013  
1                  3           13.7     1    1      1  2013  
2                  3           11.9     2    1      1  2013  


## 3.3 Chronological train/validation/test split

The paper allocates 70% of the data to training, 15% to validation, and 15% to testing
(Section IV-A). The split is performed chronologically rather than by random sampling, so that
no future information is available during training. See `docs/deviations.md` entry 4.

In [3]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

for name, part in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name:6s} {len(part):6d} rows   {part['ts'].min()}  to  {part['ts'].max()}")

Train   36804 rows   2013-01-01 00:00:00  to  2017-03-14 16:00:00
Val      7887 rows   2017-03-14 17:00:00  to  2018-02-06 07:00:00
Test     7887 rows   2018-02-06 08:00:00  to  2018-12-31 23:00:00


### Split boundaries

| Split | Rows | Period |
|---|---|---|
| Train | 36,804 | 2013-01-01 to 2017-03-14 |
| Validation | 7,887 | 2017-03-14 to 2018-02-06 |
| Test | 7,887 | 2018-02-06 to 2018-12-31 |

The test set covers a single eleven-month window (February to December 2018). Any
year-specific conditions in that period — weather, generation availability, market rule
changes — will be reflected in the reported test metrics. Walk-forward validation across
multiple test periods would address this and is noted as a candidate extension.

## 3.4 Feature scaling

The paper states that "feature normalization is employed" (Section IV-A) without specifying the
method. Min-max scaling to [0, 1] is assumed, consistent with the magnitude of the normalised
error metrics in Table 2.

The scaler is fitted on the training split only and then applied to validation and test. Fitting
on the full dataset would leak information about future value ranges into training.

In [4]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(train_df[FEATURES])

train_scaled = scaler.transform(train_df[FEATURES])
val_scaled = scaler.transform(val_df[FEATURES])
test_scaled = scaler.transform(test_df[FEATURES])

target_idx = FEATURES.index(TARGET)
print("Target column index:", target_idx)

print("\nTrain scaled range per feature:")
for i, f in enumerate(FEATURES):
    print(f"  {f:20s} min={train_scaled[:, i].min():.3f}  max={train_scaled[:, i].max():.3f}")

print("\nTest SPP scaled range: "
      f"min={test_scaled[:, target_idx].min():.3f}  max={test_scaled[:, target_idx].max():.3f}")

Target column index: 1

Train scaled range per feature:
  Load_MW              min=0.000  max=1.000
  SPP                  min=0.000  max=1.000
  temperature_2m       min=0.000  max=1.000
  dewpoint_2m          min=0.000  max=1.000
  winddirection_10m    min=0.000  max=1.000
  windspeed_10m        min=0.000  max=1.000
  hour                 min=0.000  max=1.000
  day                  min=0.000  max=1.000
  month                min=0.000  max=1.000
  year                 min=0.000  max=1.000

Test SPP scaled range: min=0.002  max=0.928
